# Course advisory: LLM runs on Kaggle GPU

Runs every LLM step of the pipeline with transformers `Qwen/Qwen2.5-1.5B-Instruct` at revision
`989aa7980e4cf806f80c7fef2b1adb7bc71aa306` (the HF original of Ollama `qwen2.5:1.5b`), on a T4.

It executes the same repo code as the local run (`src/`, `params.yaml`, `prompts/pipeline/*`), with
`LLM_BACKEND=kaggle-transformers`, so prompts, seeds, temperatures, token caps and the cache record
format are identical. Only the backend differs, and the backend is part of each cache key.

The steps come from `run_config.json`, written by `scripts/kaggle_run.sh` for one stage:
- `smoke`: 10 rows of paraphrase, 10 base-model answers, 3 augmentation paraphrases;
- `test`: query paraphrase and justification rewrite for the test split only;
- `rest`: paraphrases for all other splits, base-model answers for the train pool, then the
  120-parent augmentation paraphrase sample. Earlier Kaggle cache rows are shipped in the dataset,
  so finished calls are not repeated.

Everything lands in `/kaggle/working/kaggle_out/`, which `scripts/kaggle_run.sh` downloads and
`src/queries/merge_kaggle_cache.py` merges into `data/interim/llm_cache.jsonl`.

In [ ]:
import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

# Kaggle may mount the dataset under /kaggle/input/<slug>/ or /kaggle/input/datasets/<user>/<slug>/,
# and may unpack the uploaded tar.gz itself, so look for either form.
print(sorted(str(p) for p in Path('/kaggle/input').rglob('*') if p.is_dir())[:20])
REPO = Path('/kaggle/working/repo')
OUT = Path('/kaggle/working/kaggle_out')
OUT.mkdir(parents=True, exist_ok=True)
tarball = next(Path('/kaggle/input').rglob('repo_inputs.tar.gz'), None)
if tarball is not None:
    REPO.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tarball) as tf:
        tf.extractall(REPO)
    print('extracted', tarball)
else:
    params = min(Path('/kaggle/input').rglob('params.yaml'), key=lambda p: len(p.parts))
    shutil.copytree(params.parent, REPO, dirs_exist_ok=True)
    print('copied unpacked dataset from', params.parent)
print(sorted(p.name for p in REPO.iterdir()))

In [ ]:
import torch, transformers
assert torch.cuda.is_available(), 'no CUDA device: stop here instead of running on CPU'
env = {
    'backend': 'kaggle-transformers',
    'hf_repo': 'Qwen/Qwen2.5-1.5B-Instruct',
    'hf_revision': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306',
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
    'gpu_count': torch.cuda.device_count(),
    'python': sys.version.split()[0],
}
(OUT / 'environment.json').write_text(json.dumps(env, indent=1))
env

In [ ]:
os.environ['LLM_BACKEND'] = 'kaggle-transformers'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # one GPU, no sharding across T4 x2
timings = {}

def step(name, *args):
    t0 = time.time()
    cmd = [sys.executable, '-m', *args]
    print('>>', ' '.join(cmd), flush=True)
    with open(OUT / f'{name}.log', 'w') as log:
        p = subprocess.run(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
    timings[name] = {'seconds': round(time.time() - t0, 1), 'returncode': p.returncode}
    print(name, timings[name], flush=True)
    print(open(OUT / f'{name}.log').read()[-2000:])

In [ ]:
# run_config.json is written by scripts/kaggle_run.sh for the stage (smoke, test or rest):
# {"stage": ..., "steps": [[name, module, arg, ...], ...]}
config = json.loads((REPO / 'run_config.json').read_text())
print(config)
os.environ.update(config.get('env', {}))
for name, module, *args in config['steps']:
    step(name, module, *args)

In [ ]:
import shutil
copies = {
    'data/interim/llm_cache.jsonl': 'llm_cache.jsonl',
    'data/processed/queries_para.jsonl': 'queries_para.jsonl',
    'data/interim/base_model_answers.jsonl': 'base_model_answers.jsonl',
    'data/interim/aug_paraphrase.jsonl': 'aug_paraphrase.jsonl',
    'supporting_files/C_labelling/paraphrase_run.json': 'paraphrase_run.json',
    'supporting_files/D_active_learning/base_model_run.json': 'base_model_run.json',
    'supporting_files/D_augmentation/paraphrase_sample.json': 'paraphrase_sample.json',
}
for src, dst in copies.items():
    if (REPO / src).exists():
        shutil.copy(REPO / src, OUT / dst)
(OUT / 'timings.json').write_text(json.dumps(timings, indent=1))
shutil.rmtree(REPO)
sorted(p.name for p in OUT.iterdir())